[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-10-capstone-dashboard.ipynb#scrollTo=aa201001)

---
# Day 10 · Capstone — Interactive EDA Dashboard
**certified-journeys / altair-certified** · Day 10 · Exam Badge · Multi-Panel Gapminder Dashboard

> **Goal for today:** Build a fully interactive, multi-panel EDA dashboard from the gapminder dataset — linked brushing between scatter, histogram, and bar panels, a choropleth map, a regression trend line, and a consistent custom theme across all panels.

In [ ]:
%pip install -q altair vega-datasets

## Capstone Overview

You will build a **4-panel interactive EDA dashboard** for the gapminder dataset:

| Panel | Type | Interactions |
|---|---|---|
| 1 — Scatter | GDP per capita vs life expectancy + regression line | `selection_interval` brush source |
| 2 — Histogram | Distribution of life expectancy | Filtered by Panel 1 brush |
| 3 — Bar | Mean life expectancy by continent | Filtered by dropdown (continent) |
| 4 — Choropleth | World map colored by life expectancy | Standalone, same year filter |

**Architecture:**
```
brush = alt.selection_interval()   # defined ONCE, referenced by multiple panels

Panel 1 (scatter)  → add_params(brush)           # brush source
Panel 2 (hist)     → transform_filter(brush)     # cross-filtered by brush
Panel 3 (bar)      → transform_filter(dropdown)  # filtered by dropdown
Panel 4 (map)      → standalone (same year)

Layout: (panel1 | panel2) & (panel3 | panel4)
```

**Strategy:** Build and verify each panel independently, then wire interactions and compose last.

## Step 1 · Load and Inspect the Data

In [ ]:
import altair as alt
import pandas as pd
from vega_datasets import data as vd

# Load the gapminder dataset
gap_all = vd.gapminder()

print("Shape:", gap_all.shape)
print("Columns:", gap_all.columns.tolist())
print("\nYears available:", sorted(gap_all['year'].unique()))
print("\nSample (year=2000):")
gap_2000 = gap_all[gap_all['year'] == 2000].copy()
print(gap_2000.head(8))

**What just happened?**

- The gapminder dataset has country-level annual data: population, fertility, life expectancy, GDP proxy (via cluster), and geographic info.
- We'll filter to `year == 2000` for a single-year cross-sectional analysis.
- `cluster` (0–5) is a World Bank income-group proxy for continent/region — we use it as a categorical color dimension.
- `fertility` and `life_expect` are the core EDA dimensions for the scatter panel.

## Step 2 · Define the Custom Theme

Register a consistent theme before building any panel. All 4 panels will inherit these defaults.

In [ ]:
def gapminder_dashboard_theme():
    """Clean, print-friendly theme for the gapminder dashboard."""
    font = "Helvetica Neue, Arial, sans-serif"
    return {
        "config": {
            "background": "#FFFFFF",
            "view": {"stroke": "transparent"},
            "title": {
                "font": font, "fontSize": 13, "fontWeight": 600,
                "color": "#1a1a2e", "anchor": "start", "offset": 4,
            },
            "axis": {
                "labelFont": font, "labelFontSize": 10,
                "titleFont": font, "titleFontSize": 11,
                "gridColor": "#eeeeee", "gridWidth": 0.75,
                "domainColor": "#cccccc", "tickColor": "#cccccc",
            },
            "legend": {
                "labelFont": font, "labelFontSize": 10,
                "titleFont": font, "titleFontSize": 10, "titleFontWeight": 600,
            },
            "range": {
                # 6-color categorical palette for income clusters 0–5
                "category": ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]
            },
        }
    }

# Register and enable the theme for all subsequent charts
alt.theme.register(gapminder_dashboard_theme, name="gapminder-dashboard")
alt.theme.enable("gapminder-dashboard")
print("Theme registered and enabled.")

**What just happened?**

- The theme is registered once and applies globally to all `alt.Chart` calls in this session.
- We define 6 categorical colors matching the 6 income clusters (0–5) in the gapminder dataset.
- `view.stroke: transparent` removes the chart border for a cleaner dashboard look.
- `anchor: 'start'` left-aligns all chart titles — consistent with modern dashboard conventions.

## Step 3 · Panel 1 — Scatter Plot with Brush + Regression Line

The scatter plot is the **brush source** — it generates the selection that all other panels respond to. We also layer a `transform_regression` trend line over the scatter.

In [ ]:
# Filter to one year
gap_2000 = gap_all[gap_all['year'] == 2000].copy()

# Define the shared brush selection — x-axis only for cleaner UX
# Note: we use encodings=['x'] to brush only along life_expect
brush = alt.selection_interval(
    encodings=["x"],   # restrict brush to x-axis (life_expect)
    name="brush",      # name the selection for clarity in composed charts
)

# Scatter points
scatter_points = (
    alt.Chart(gap_2000)
    .mark_circle(size=80, opacity=0.8)
    .encode(
        x=alt.X("life_expect:Q", title="Life Expectancy (years)",
                scale=alt.Scale(domain=[30, 90])),
        y=alt.Y("fertility:Q", title="Fertility Rate",
                scale=alt.Scale(domain=[0, 9])),
        color=alt.condition(
            brush,
            alt.Color("cluster:O", title="Income Group",
                      scale=alt.Scale(scheme="category10")),
            alt.value("#cccccc")
        ),
        size=alt.condition(brush, alt.value(80), alt.value(25)),
        tooltip=[
            alt.Tooltip("country:N"),
            alt.Tooltip("life_expect:Q", format=".1f"),
            alt.Tooltip("fertility:Q", format=".2f"),
            alt.Tooltip("pop:Q", title="Population", format=",.0f"),
        ],
    )
    .add_params(brush)   # scatter is the brush SOURCE
)

# Regression trend line layered on the scatter
regression_line = (
    alt.Chart(gap_2000)
    .mark_line(color="#E8890C", strokeWidth=2.5, strokeDash=[6, 3])
    .transform_regression(
        on="life_expect",    # x field
        regression="fertility",  # y field
        method="linear",
    )
    .encode(
        x="life_expect:Q",
        y="fertility:Q",
    )
)

# Compose scatter + regression
panel1 = (
    alt.layer(scatter_points, regression_line)
    .properties(
        title="Life Expectancy vs Fertility Rate (2000) — Drag to brush",
        width=420,
        height=280,
    )
)

# Test panel 1 in isolation
panel1

**What just happened?**

- `alt.selection_interval(encodings=['x'])` creates a brush restricted to the x-axis — dragging selects a life-expectancy range.
- `transform_regression(on='life_expect', regression='fertility')` computes a least-squares linear fit and renders it as a line.
- **The regression line uses the full dataset** — not the brushed subset — so it's a stable reference.
- `alt.layer(scatter, regression)` stacks the trend line on top of the scatter points.

## Step 4 · Panel 2 — Histogram Filtered by Brush

In [ ]:
# Histogram of life_expect — filtered by the brush from Panel 1
panel2 = (
    alt.Chart(gap_2000)
    .mark_bar(color="#4C72B0", opacity=0.8)
    .encode(
        x=alt.X(
            "life_expect:Q",
            bin=alt.Bin(maxbins=20),
            title="Life Expectancy",
        ),
        y=alt.Y("count():Q", title="Number of Countries"),
        tooltip=[
            alt.Tooltip("life_expect:Q", bin=True, title="Life Expect Range"),
            alt.Tooltip("count():Q", title="Countries"),
        ],
    )
    .transform_filter(brush)   # only show bars for brushed countries
    .properties(
        title="Distribution of Life Expectancy (brushed countries)",
        width=300,
        height=280,
    )
)

# Test panel 2 in isolation (will show all bars until brush is active)
panel2

**What just happened?**

- `transform_filter(brush)` uses the same `brush` selection object defined in Panel 1.
- **When viewed in isolation the histogram shows all countries** — the brush is inactive until Panel 1 is also rendered and a rectangle is drawn.
- Once composed side-by-side with Panel 1, dragging on the scatter updates this histogram in real time.
- `alt.Bin(maxbins=20)` lets Vega-Lite choose sensible bin boundaries automatically.

## Step 5 · Panel 3 — Grouped Bar with Dropdown Filter

In [ ]:
# Build cluster-to-label mapping for the dropdown
cluster_labels = {
    0: "South Asia", 1: "Europe & C. Asia",
    2: "Sub-Saharan Africa", 3: "Americas",
    4: "E. Asia & Pacific", 5: "Mid. East & N. Africa"
}

# Add a readable cluster name column
gap_2000 = gap_2000.copy()
gap_2000["region"] = gap_2000["cluster"].map(cluster_labels).fillna("Unknown")

# Dropdown widget: filter by region
regions = sorted(gap_2000["region"].unique().tolist())
region_binding = alt.binding_select(
    options=[None] + regions,
    labels=["All Regions"] + regions,
    name="Region: "
)
region_sel = alt.selection_point(
    fields=["region"],
    bind=region_binding,
    name="region_filter"
)

panel3 = (
    alt.Chart(gap_2000)
    .mark_bar()
    .encode(
        x=alt.X("mean(life_expect):Q", title="Mean Life Expectancy",
                scale=alt.Scale(domain=[40, 85])),
        y=alt.Y("region:N", sort="-x", title="Region"),
        color=alt.Color("region:N", legend=None),
        tooltip=[
            alt.Tooltip("region:N"),
            alt.Tooltip("mean(life_expect):Q", format=".1f", title="Mean Life Expect"),
            alt.Tooltip("count():Q", title="Countries"),
        ],
    )
    .transform_filter(region_sel)   # dropdown filter
    .add_params(region_sel)
    .properties(
        title="Mean Life Expectancy by Region — Filter with dropdown",
        width=380,
        height=220,
    )
)

# Test panel 3 in isolation
panel3

**What just happened?**

- `binding_select` creates a dropdown widget above the chart — `options=[None]` + label `'All Regions'` enables the "show all" state.
- `transform_filter(region_sel)` hides rows that don't match the selected region.
- When `None` is selected ("All Regions"), the selection is empty and all rows pass the filter.
- `sort='-x'` sorts the bars by descending mean life expectancy — the best-performing region is always at the top.

## Step 6 · Panel 4 — Choropleth Map by Life Expectancy

In [ ]:
# World map choropleth colored by life_expect
# The gapminder dataset has a numeric country id field
world_url = vd.world_110m.url

# Inspect: does gapminder have a numeric id we can join on?
print(gap_2000.dtypes)
print("\nSample id-like columns:", gap_2000[['country', 'cluster']].head(3))

In [ ]:
# The world_110m topology uses ISO-3166 numeric country IDs in the 'id' property.
# The gapminder dataset from vega-datasets doesn't have a numeric id directly.
# We can join on country name if the gapminder_url data is used directly as a URL source.
# For robustness, use the data URL directly in Altair (avoids Python-side join).

gapminder_url = vd.gapminder.url  # CDN URL — Vega-Lite fetches it directly

# Use a filter transform to restrict to year 2000
panel4 = (
    alt.Chart(alt.topo_feature(world_url, "countries"))
    .mark_geoshape(stroke="white", strokeWidth=0.3)
    .encode(
        color=alt.Color(
            "life_expect:Q",
            title="Life Expectancy",
            scale=alt.Scale(scheme="viridis", domain=[40, 85]),
        ),
        tooltip=[
            alt.Tooltip("country:N", title="Country"),
            alt.Tooltip("life_expect:Q", format=".1f", title="Life Expect"),
        ],
    )
    .transform_lookup(
        lookup="id",   # numeric country id in the TopoJSON
        from_=alt.LookupData(
            alt.UrlData(gapminder_url),   # load gapminder from CDN
            "id",                          # matching field in gapminder
            ["life_expect", "country", "year"]
        )
    )
    .transform_filter("datum.year == 2000")  # filter to year 2000 after join
    .project("naturalEarth1")
    .properties(
        title="Life Expectancy by Country (2000)",
        width=680,
        height=340,
    )
)

# Test panel 4 in isolation
panel4

**What just happened?**

- `alt.UrlData(gapminder_url)` tells Altair to load the data from the CDN URL at render time — this works in Colab and browser-based rendering.
- `transform_filter("datum.year == 2000")` is a **Vega expression string** — it's evaluated inside Vega-Lite, not in Python.
- `scale=alt.Scale(scheme='viridis', domain=[40, 85])` pins the color scale range so the map is comparable across filter changes.
- Countries with no matching data (due to name mismatches) appear as the default grey fill.

## Step 7 · Compose the Full Dashboard

In [ ]:
# Compose all 4 panels into a single dashboard
# Layout:
#   Top row:    panel1 (scatter+brush)  |  panel2 (histogram, filtered by brush)
#   Bottom row: panel3 (bar+dropdown)   |  [spacer — panel4 is full width below]
#   Full width: panel4 (choropleth map)

top_row = panel1 | panel2

full_dashboard = alt.vconcat(
    top_row,
    panel3,
    panel4,
).properties(
    title=alt.TitleParams(
        text="Gapminder 2000 — Interactive EDA Dashboard",
        subtitle="Brush the scatter to filter the histogram · Use dropdown to filter the bar chart",
        fontSize=16,
        subtitleFontSize=11,
        anchor="start",
        color="#1a1a2e",
        subtitleColor="#666",
    )
).configure_view(
    strokeWidth=0,
)

full_dashboard

**What just happened?**

- `panel1 | panel2` is shorthand for `alt.hconcat(panel1, panel2)` — horizontal composition.
- `alt.vconcat(top_row, panel3, panel4)` stacks all rows vertically into a single Vega-Lite spec.
- `alt.TitleParams` allows a subtitle — useful for explaining the interaction model to new users.
- **The brush interaction works across panels because `brush` was defined once and referenced by name** — Vega-Lite propagates the selection across all views in the composed spec.
- `configure_view(strokeWidth=0)` removes the thin frame borders for a cleaner dashboard look.

In [ ]:
# ============================================================
# CAPSTONE CHALLENGE
# Build a fully interactive multi-panel EDA dashboard
# from the gapminder Vega dataset.
# ============================================================
#
# Your task: extend and enhance the dashboard built in Steps 3–7
#
# REQUIRED ADDITIONS:
# ------------------------------------------------------------------
#
# 1. PANEL 1 — Scatter (GDP proxy vs life_expect)
#    - Currently uses fertility on Y. Change Y to use 'pop' (population)
#      on a log scale: alt.Y('pop:Q', scale=alt.Scale(type='log'))
#    - Add a YEAR slider using alt.binding_range to animate over years
#      HINT: define a year_sel = alt.selection_point(bind=alt.binding_range(
#              min=1955, max=2005, step=5, name='Year: '), fields=['year'])
#            then use transform_filter(year_sel) on the scatter data
#
# 2. PANEL 2 — Histogram
#    - Currently shows life_expect bins. Add a second y-encoding:
#      stack the bars by cluster (income group) using
#      color=alt.Color('cluster:O') with mark_bar stacking
#
# 3. PANEL 3 — Bar chart
#    - Add error bars showing the min and max life_expect per region
#      HINT: use mark_rule() as a second layer with
#      x=alt.X('min(life_expect):Q'), x2='max(life_expect):Q'
#
# 4. PANEL 4 — Choropleth
#    - Wire the year_sel slider from Panel 1 to the choropleth
#      so the map also updates when the year changes
#      HINT: add .add_params(year_sel) to Panel 4 and use
#            transform_filter(year_sel) instead of the hard-coded 2000 filter
#
# 5. THEME
#    - Ensure the custom theme is active (call alt.theme.enable(...))
#    - Add a consistent title to every panel
#
# 6. LAYOUT
#    - Arrange as: (scatter | histogram) stacked above (bar | map)
#    - Add a title to the composed dashboard with a subtitle
#      explaining the linked interactions
#
# ------------------------------------------------------------------
# SCAFFOLD (fill in the sections marked TODO)
# ------------------------------------------------------------------

from vega_datasets import data as vd
import altair as alt

# --- Data ---
gap_all = vd.gapminder()
world_url = vd.world_110m.url
gapminder_url = vd.gapminder.url

# --- Ensure custom theme is active ---
alt.theme.enable("gapminder-dashboard")  # registered earlier in this notebook

# --- Selections ---
# Brush: interval on x-axis (life_expect), filters histogram
brush = alt.selection_interval(encodings=["x"], name="brush")

# TODO: Year slider selection (covers all years in gapminder: 1955–2005 in steps of 5)
# year_sel = alt.selection_point(
#     fields=['year'],
#     bind=alt.binding_range(min=1955, max=2005, step=5, name='Year: '),
#     name='year_filter',
#     value=2000,   # default to year 2000
# )

# TODO: Region dropdown (same as Step 5 above)
# region_binding = ...
# region_sel = ...

# --- Panel 1: Scatter (life_expect vs pop log scale) + regression ---
# TODO: scatter_points = (
#     alt.Chart(gap_all)     # use gap_all so the slider can change years
#     .mark_circle(size=70, opacity=0.75)
#     .encode(
#         x=alt.X('life_expect:Q', title='Life Expectancy', scale=alt.Scale(domain=[30, 90])),
#         y=alt.Y('pop:Q', title='Population', scale=alt.Scale(type='log')),
#         color=alt.condition(brush, alt.Color('cluster:O'), alt.value('#ccc')),
#         tooltip=['country:N', 'life_expect:Q', 'pop:Q', 'year:O'],
#     )
#     .transform_filter(year_sel)   # filter by year slider
#     .add_params(brush, year_sel)  # scatter owns both selections
#     .properties(title='Life Expect vs Population — drag to brush', width=400, height=260)
# )
#
# TODO: regression_line = ( ... .transform_regression(...) ... )
# TODO: panel1 = alt.layer(scatter_points, regression_line)

# --- Panel 2: Stacked histogram filtered by brush ---
# TODO: panel2 = (
#     alt.Chart(gap_all)
#     .mark_bar()
#     .encode(
#         x=alt.X('life_expect:Q', bin=alt.Bin(maxbins=20), title='Life Expectancy'),
#         y=alt.Y('count():Q'),
#         color=alt.Color('cluster:O', title='Income Group'),
#     )
#     .transform_filter(brush)
#     .transform_filter(year_sel)
#     .properties(title='Distribution (brushed countries)', width=280, height=260)
# )

# --- Panel 3: Bar chart with error bars ---
# TODO: bar_base = (
#     alt.Chart(gap_all)
#     .mark_bar()
#     .encode(
#         x=alt.X('mean(life_expect):Q', title='Mean Life Expectancy', scale=alt.Scale(domain=[0,90])),
#         y=alt.Y('cluster:O', sort='-x'),
#         color=alt.Color('cluster:O', legend=None),
#     )
#     .transform_filter(region_sel)
#     .transform_filter(year_sel)
#     .add_params(region_sel)
# )
#
# TODO: error_bars = (
#     alt.Chart(gap_all)
#     .mark_rule(color='#333')
#     .encode(
#         x=alt.X('min(life_expect):Q'),
#         x2='max(life_expect):Q',
#         y=alt.Y('cluster:O'),
#     )
#     .transform_filter(region_sel)
#     .transform_filter(year_sel)
# )
# TODO: panel3 = alt.layer(bar_base, error_bars).properties(title='...', width=360, height=220)

# --- Panel 4: Choropleth wired to year slider ---
# TODO: panel4 = (
#     alt.Chart(alt.topo_feature(world_url, 'countries'))
#     .mark_geoshape(stroke='white', strokeWidth=0.3)
#     .encode(
#         color=alt.Color('life_expect:Q', scale=alt.Scale(scheme='viridis', domain=[40,85])),
#         tooltip=['country:N', 'life_expect:Q'],
#     )
#     .transform_lookup(
#         lookup='id',
#         from_=alt.LookupData(alt.UrlData(gapminder_url), 'id', ['life_expect', 'country', 'year'])
#     )
#     .transform_filter(year_sel)    # wired to the same year slider!
#     .add_params(year_sel)
#     .project('naturalEarth1')
#     .properties(title='Life Expectancy Map', width=640, height=300)
# )

# --- Compose ---
# TODO: dashboard = alt.vconcat(
#     panel1 | panel2,
#     panel3 | panel4,
# ).properties(title=alt.TitleParams(
#     text='Gapminder Interactive EDA Dashboard',
#     subtitle='Brush scatter to filter histogram · Slider changes year across all panels',
#     anchor='start',
# ))
#
# dashboard

print("Capstone scaffold ready — uncomment and fill in the TODO sections above.")
print("Build and test each panel independently, then compose last.")
print("\nGapminder years available:", sorted(gap_all['year'].unique()))

---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| Multi-panel shared selection | Define `brush` once; reference it in `add_params` and `transform_filter` across panels |
| `transform_regression` | Computes OLS regression inside Vega-Lite — no scikit-learn needed |
| Year slider | `alt.selection_point(bind=alt.binding_range(...))` — filter all panels consistently |
| `alt.vconcat` / `\|` | Vertical / horizontal composition; all panels share a single Vega-Lite spec |
| `alt.TitleParams` | Rich title with subtitle, font, anchor |
| Choropleth + URL data | `alt.UrlData(url)` for remote data sources; `transform_filter(year_sel)` for cross-panel sync |
| Theme registration | `alt.theme.register` + `alt.theme.enable` — one call before any chart, global effect |

> **Tip:** Build and test each panel independently before wiring linked selections — debug in isolation, compose last. Use `|` and `&` to join panels, then wrap in `alt.hconcat()` for final layout control.

---
## What's next
**Congratulations — you've completed the Vega-Altair for Data Visualization course!** 🎉

You've built interactive selections, custom themes, geographic maps, and a full multi-panel EDA dashboard. Continue exploring:

- [Altair Gallery](https://altair-viz.github.io/gallery/index.html) — hundreds of examples
- [Vega-Lite Spec](https://vega.github.io/vega-lite/) — the underlying spec Altair compiles to
- [Panel + Altair](https://panel.holoviz.org/) — deploy Altair dashboards as web apps

Mark Day 10 complete in your [tracker](../index.html).